In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/ab_data.csv')

In [3]:
print(df.shape)
display(df.head())

(294478, 5)


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


In [5]:
# 設定が矛盾しているデータ
mismatch = df[((df["group"]=="treatment")==(df["landing_page"]=="new_page"))==False].shape[0]
print(mismatch)

# 正しい組み合わせのものだけ残す
df2 = df[((df['group']=='treatment')==(df['landing_page']=='new_page'))].copy()

# 重複アクセスを削除
df2 = df2.drop_duplicates(subset="user_id",keep="first")
print(df2.shape[0])

# グループごとの購入率
conversion_rates = df2.groupby("group")["converted"].mean()*100
print(conversion_rates)

3893
290584
group
control      12.038630
treatment    11.880807
Name: converted, dtype: float64


In [7]:
import scipy.stats as stats

# クロス集計表の作成
contingency_table = pd.crosstab(df2["group"],df2["converted"])
display(contingency_table)

# カイ二乗検定
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print(f"\nカイ二乗値: {chi2:.4f}")
print(f"p値 (p-value): {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
  print("結論: p値が0.05より小さいため、新旧ページでコンバージョン率に統計的な有意差があります。")
else:
  print("結論: p値が0.05以上のため、コンバージョン率の差は「誤差の範囲内」であり、有意差はありません。")

converted,0,1
group,,
control,127785,17489
treatment,128046,17264



カイ二乗値: 1.7036
p値 (p-value): 0.1918
結論: p値が0.05以上のため、コンバージョン率の差は「誤差の範囲内」であり、有意差はありません。


In [9]:
import statsmodels.api as sm

df2["ab_page"] = pd.get_dummies(df2["group"],drop_first=True).astype(int)
df2["intercept"] = 1

# ロジスティック回帰
logit_mod = sm.Logit(df2["converted"],df2[["intercept","ab_page"]])
results = logit_mod.fit()

print(results.summary2())

Optimization terminated successfully.
         Current function value: 0.366118
         Iterations 6
                          Results: Logit
Model:              Logit            Method:           MLE        
Dependent Variable: converted        Pseudo R-squared: 0.000      
Date:               2026-03-19 11:40 AIC:              212780.3502
No. Observations:   290584           BIC:              212801.5095
Df Model:           1                Log-Likelihood:   -1.0639e+05
Df Residuals:       290582           LL-Null:          -1.0639e+05
Converged:          1.0000           LLR p-value:      0.18988    
No. Iterations:     6.0000           Scale:            1.0000     
-------------------------------------------------------------------
              Coef.   Std.Err.      z      P>|z|    [0.025   0.975]
-------------------------------------------------------------------
intercept    -1.9888    0.0081  -246.6690  0.0000  -2.0046  -1.9730
ab_page      -0.0150    0.0114    -1.3109  0.1899